# 物理层安全 — IoT设备认证

## 为什么需要物理层安全?

传统安全依赖上层加密 (密码学), 但IoT设备面临:
- 资源受限: 算力不够运行复杂加密算法
- 密钥管理难: 大规模部署时密钥分发困难
- 身份伪造: 攻击者可以克隆MAC地址等上层标识

物理层安全的思路: **利用无线信道的物理特性作为安全凭证**
- 射频指纹: 设备硬件的唯一特征
- 信道特征: 通信双方的唯一信道响应

```
传统安全:  你是谁? -> 密码/证书 (可被窃取/克隆)
物理层安全: 你是谁? -> 你的物理特征 (不可克隆)
```

## 本课内容
1. 物理层安全基础
2. IoT认证威胁模型
3. 射频指纹认证
4. 信道特征密钥生成
5. 认证协议设计
6. 攻击与防御
7. 实际系统与标准

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix
import seaborn as sns
np.random.seed(42)

## 1. 物理层安全基础

### 三大核心原理

```
1. 互易性 (Reciprocity)
   A->B 和 B->A 的信道响应相同 (同时同频)
   -> 合法双方可提取相同的信道特征

2. 空间唯一性 (Spatial Uniqueness)
   不同位置的信道响应不同
   -> 窃听者无法获得合法用户的信道特征

3. 硬件唯一性 (Hardware Uniqueness)
   每个设备的射频指纹不同
   -> 可用于设备身份认证
```

### 物理层安全的两大方向

| 方向 | 原理 | 目标 |
|------|------|------|
| 射频指纹认证 | 硬件唯一性 | 识别设备身份 |
| 信道特征密钥生成 | 互易性+空间唯一性 | 生成共享密钥 |

In [ ]:
# 互易性演示
n_channels = 5
h_AB = np.random.randn(n_channels) + 1j * np.random.randn(n_channels)
h_BA = h_AB + 0.01 * (np.random.randn(n_channels) + 1j * np.random.randn(n_channels))
h_AE = np.random.randn(n_channels) + 1j * np.random.randn(n_channels)

corr_AB = np.abs(np.corrcoef(h_AB.real, h_BA.real)[0, 1])
corr_AE = np.abs(np.corrcoef(h_AB.real, h_AE.real)[0, 1])

print("=== 信道互易性 ===")
print(f"h_AB 与 h_BA 的相关性: {corr_AB:.4f} (接近1, 互易)")
print(f"h_AB 与 h_AE 的相关性: {corr_AE:.4f} (接近0, 不同)")
print()
print("Alice和Bob可以从信道提取相同特征, 而Eve不行")

In [ ]:
# 硬件唯一性演示
devices = {
    'Device A': {'CFO': 500, 'IQ_amp': 1.05, 'IQ_phase': 3, 'PA': 0.03},
    'Device B': {'CFO': -300, 'IQ_amp': 0.98, 'IQ_phase': -2, 'PA': 0.07},
    'Device C': {'CFO': 100, 'IQ_amp': 1.10, 'IQ_phase': 5, 'PA': 0.01},
    'Clone A':  {'CFO': 500, 'IQ_amp': 1.05, 'IQ_phase': 3, 'PA': 0.03},
}

print("=== 硬件唯一性 (射频指纹) ===")
print()
for name, p in devices.items():
    print(f"{name}: CFO={p['CFO']}Hz, IQ_amp={p['IQ_amp']}, IQ_phase={p['IQ_phase']}deg, PA={p['PA']}")
print()
print("即使克隆了参数, 实际中无法精确复制模拟电路的随机缺陷")
print("射频指纹是物理不可克隆函数 (PUF) 的一种体现")

---
## 2. IoT认证威胁模型

### 典型攻击场景

```
                    +----------+
         +--------->|   Eve    |
         | 窃听     |(Eavesdrop)|
         v          +----------+
+------+  无线信道  +------+
| Alice |<--------->|  Bob  |
| (IoT) |           | (网关) |
+------+           +------+
         ^
         | 伪造/重放
         |
    +----------+
    | Mallory  |
    |(Spoofing)|
    +----------+
```

### 攻击类型

| 攻击 | 描述 | 危害 |
|------|------|------|
| 窃听 | 被动监听通信 | 信息泄露 |
| 伪造 | 伪造合法设备身份 | 非法接入 |
| 重放 | 重放合法设备的信号 | 绕过认证 |
| 中间人 | 拦截并篡改通信 | 数据篡改 |
| 拒绝服务 | 干扰通信 | 服务中断 |
| 克隆 | 复制设备身份信息 | 身份盗用 |

In [ ]:
# 传统认证 vs 物理层认证
print("=== 传统认证 vs 物理层认证 ===")
print()
dims = ["凭证", "可克隆性", "计算开销", "密钥管理", "抗重放", "抗伪造", "准确率", "适用场景"]
trad = ["密码/证书/密钥", "可被复制", "高(加密运算)", "需要", "需要nonce", "依赖密码强度", "100%(理想)", "所有场景"]
phys = ["射频指纹/信道特征", "物理不可克隆", "低(特征提取)", "不需要/轻量", "天然抗重放", "依赖物理不可克隆性", "<100%(统计)", "无线通信"]

for d, t, p in zip(dims, trad, phys):
    print(f"  {d:<10} | 传统: {t:<20} | 物理层: {p}")
print()
print("最佳实践: 物理层认证 + 传统认证 = 多因子认证")
print("  物理层作为第一道防线(快速筛选)")
print("  传统方法作为第二道防线(精确验证)")

---
## 3. 射频指纹认证

### 认证流程

```
注册阶段 (Enrollment):
  设备 -> 采集IQ信号 -> 提取指纹 -> 存入数据库

认证阶段 (Authentication):
  设备 -> 采集IQ信号 -> 提取指纹 -> 与数据库比对 -> 认证结果
```

### 两种认证模式

1. **Closed-set**: 已知设备库, 判断是哪个设备
2. **Open-set**: 判断是否为已知设备 (拒绝未知设备)

In [ ]:
# 模拟IoT设备的射频指纹
def generate_iot_fingerprint(device_id, n_samples=128, snr=20):
    t = np.linspace(0, 1, n_samples)
    np.random.seed(device_id * 77 + 13)
    I = np.random.choice([-1, 1], n_samples) / np.sqrt(2)
    Q = np.random.choice([-1, 1], n_samples) / np.sqrt(2)
    s = I + 1j * Q
    cfo = (device_id - 5) * 200 + np.random.randn() * 50
    amp_imb = 1.0 + (device_id % 7 - 3) * 0.04
    phase_imb = np.deg2rad((device_id % 5 - 2) * 3)
    pa_coeff = 0.02 + (device_id % 3) * 0.02
    s = s * np.exp(1j * 2 * np.pi * cfo * t)
    I_new = s.real * amp_imb
    Q_new = s.imag * np.cos(phase_imb) + s.real * np.sin(phase_imb)
    s = I_new + 1j * Q_new
    s = s - pa_coeff * s * np.abs(s)**2
    noise_power = np.mean(np.abs(s)**2) / (10**(snr/10))
    noise = np.sqrt(noise_power/2) * (np.random.randn(n_samples) + 1j*np.random.randn(n_samples))
    r = s + noise
    iq = np.stack([r.real, r.imag], axis=0).astype(np.float32)
    return iq

n_devices = 20
samples_per_device = 100
iq_length = 128

X_all, y_all = [], []
for dev_id in range(n_devices):
    for _ in range(samples_per_device):
        iq = generate_iot_fingerprint(dev_id, iq_length, snr=15)
        X_all.append(iq)
        y_all.append(dev_id)

X_all = np.array(X_all)
y_all = np.array(y_all)
X_all = X_all / (np.std(X_all) + 1e-8)

print(f"数据集: {X_all.shape}, 设备数: {n_devices}")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.3, random_state=42, stratify=y_all)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [ ]:
# 认证模型: 1D CNN + 度量学习
class AuthNet(nn.Module):
    def __init__(self, embed_dim=64, n_classes=20):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(2, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.embedding = nn.Sequential(nn.Linear(256, embed_dim), nn.ReLU())
        self.classifier = nn.Linear(embed_dim, n_classes)
    
    def forward(self, x):
        feat = self.features(x).view(x.size(0), -1)
        embed = self.embedding(feat)
        logits = self.classifier(embed)
        return logits, embed

model = AuthNet(embed_dim=64, n_classes=n_devices)
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 训练认证模型
criterion_cls = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

train_accs, test_accs = [], []
for epoch in range(50):
    model.train()
    correct, total = 0, 0
    for X_b, y_b in train_loader:
        logits, embed = model(X_b)
        loss = criterion_cls(logits, y_b)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        correct += (logits.argmax(1) == y_b).sum().item()
        total += X_b.size(0)
    train_accs.append(correct / total)
    scheduler.step()
    
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_b, y_b in test_loader:
            logits, _ = model(X_b)
            correct += (logits.argmax(1) == y_b).sum().item()
            total += X_b.size(0)
    test_accs.append(correct / total)

plt.figure(figsize=(8, 4))
plt.plot(train_accs, label="Train")
plt.plot(test_accs, label="Test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Authentication Model Training")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print(f"测试准确率: {test_accs[-1]:.4f}")

In [ ]:
# Open-set 认证: 拒绝未知设备
X_unknown, y_unknown = [], []
for dev_id in range(n_devices, n_devices + 10):
    for _ in range(50):
        iq = generate_iot_fingerprint(dev_id, iq_length, snr=15)
        X_unknown.append(iq)
        y_unknown.append(dev_id)

X_unknown = np.array(X_unknown)
X_unknown = X_unknown / (np.std(X_unknown) + 1e-8)

model.eval()
with torch.no_grad():
    logits_known, embed_known = model(torch.from_numpy(X_test))
    probs_known = torch.softmax(logits_known, dim=1)
    max_prob_known = probs_known.max(dim=1).values.numpy()
    
    logits_unknown, embed_unknown = model(torch.from_numpy(X_unknown))
    probs_unknown = torch.softmax(logits_unknown, dim=1)
    max_prob_unknown = probs_unknown.max(dim=1).values.numpy()

y_true = np.concatenate([np.ones(len(max_prob_known)), np.zeros(len(max_prob_unknown))])
y_score = np.concatenate([max_prob_known, max_prob_unknown])
fpr, tpr, thresholds = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(max_prob_known, bins=30, alpha=0.6, label="Known (Legitimate)", color="blue", density=True)
axes[0].hist(max_prob_unknown, bins=30, alpha=0.6, label="Unknown (Attacker)", color="red", density=True)
axes[0].set_xlabel("Max Softmax Probability")
axes[0].set_ylabel("Density")
axes[0].set_title("Open-set Authentication")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(fpr, tpr, "b-", linewidth=2, label=f"AUC = {roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[1].set_xlabel("False Positive Rate (FAR)")
axes[1].set_ylabel("True Positive Rate (GAR)")
axes[1].set_title("ROC Curve for Authentication")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"AUROC: {roc_auc:.4f}")
print()
print("认证系统关键指标:")
print("  GAR (Genuine Accept Rate): 合法设备被接受的概率")
print("  FAR (False Accept Rate):   非法设备被接受的概率")
print("  FRR (False Reject Rate):   合法设备被拒绝的概率 = 1 - GAR")
print("  EER (Equal Error Rate):    FAR = FRR 时的错误率")

In [ ]:
# 度量学习: 更好的Open-set认证
print("=== 度量学习用于认证 ===")
print()
print("分类损失 (CrossEntropy) 的问题:")
print("  只关心类间可分, 不关心嵌入空间的结构")
print("  未知设备可能被强行分到某个已知类")
print()
print("度量学习的思路:")
print("  学习一个嵌入空间, 同类样本靠近, 不同类样本远离")
print("  用距离判断身份, 天然支持Open-set")
print()
print("常用度量学习损失:")
print("  1. Contrastive Loss: 拉近正对, 推开负对")
print("  2. Triplet Loss: anchor-positive < anchor-negative + margin")
print("  3. Center Loss: 每个类学一个中心, 拉近样本到中心")
print("  4. ArcFace: 角度间隔, 增大类间角度")
print()
print("RFFI推荐: ArcFace + Center Loss 组合")

In [ ]:
# ArcFace 损失实现
class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.s = s
        self.m = m
    
    def forward(self, embedding, labels):
        embedding = F.normalize(embedding, dim=1)
        weight = F.normalize(self.weight, dim=1)
        cosine = F.linear(embedding, weight)
        one_hot = F.one_hot(labels, cosine.size(1)).float()
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        target_logits = torch.cos(theta + self.m)
        output = cosine * (1 - one_hot) + target_logits * one_hot
        output *= self.s
        return output

class CenterLoss(nn.Module):
    def __init__(self, n_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(n_classes, feat_dim))
    
    def forward(self, features, labels):
        batch_size = features.size(0)
        distmat = torch.cdist(features, self.centers)**2
        losses = distmat[range(batch_size), labels]
        return losses.mean()

arcface = ArcFaceLoss(64, n_devices)
center_loss = CenterLoss(n_devices, 64)
print(f"ArcFace参数量: {sum(p.numel() for p in arcface.parameters()):,}")
print(f"Center Loss参数量: {sum(p.numel() for p in center_loss.parameters()):,}")

In [ ]:
# 用ArcFace训练认证模型
model_arc = AuthNet(embed_dim=64, n_classes=n_devices)
arcface_loss = ArcFaceLoss(64, n_devices, s=30.0, m=0.5)
center = CenterLoss(n_devices, 64)

optimizer = optim.AdamW(list(model_arc.parameters()) + list(arcface_loss.parameters()), lr=1e-3)
optimizer_center = optim.SGD(center.parameters(), lr=0.5)
ce_loss = nn.CrossEntropyLoss()

for epoch in range(50):
    model_arc.train()
    for X_b, y_b in train_loader:
        logits, embed = model_arc(X_b)
        embed_norm = F.normalize(embed, dim=1)
        arc_logits = arcface_loss(embed_norm, y_b)
        loss_cls = ce_loss(arc_logits, y_b)
        loss_center = center(embed, y_b)
        loss = loss_cls + 0.01 * loss_center
        optimizer.zero_grad()
        optimizer_center.zero_grad()
        loss.backward()
        optimizer.step()
        for param in center.parameters():
            param.grad.data *= 0.01
        optimizer_center.step()

print("ArcFace模型训练完成")

In [ ]:
# 可视化嵌入空间 (t-SNE)
from sklearn.manifold import TSNE

model_arc.eval()
with torch.no_grad():
    _, embed_test = model_arc(torch.from_numpy(X_test))
    _, embed_unknown = model_arc(torch.from_numpy(X_unknown))

embed_all = torch.cat([embed_test, embed_unknown], dim=0).numpy()
labels_all = np.concatenate([y_test, np.full(len(y_unknown), -1)])

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embed_2d = tsne.fit_transform(embed_all)

fig, ax = plt.subplots(figsize=(10, 8))
for dev_id in range(min(n_devices, 10)):
    mask = labels_all == dev_id
    ax.scatter(embed_2d[mask, 0], embed_2d[mask, 1], s=10, alpha=0.5, label=f"Dev {dev_id}")

mask_unknown = labels_all == -1
ax.scatter(embed_2d[mask_unknown, 0], embed_2d[mask_unknown, 1], s=10, alpha=0.5, 
           c="black", marker="x", label="Unknown")

ax.set_title("Embedding Space (t-SNE) - ArcFace")
ax.legend(fontsize=7, ncol=3)
plt.tight_layout()
plt.show()
print("同类设备聚集在一起, 不同设备分开")
print("未知设备(黑色x)应该远离所有已知设备簇")

---
## 4. 信道特征密钥生成

利用信道互易性, 通信双方从信道响应中提取相同的密钥。

```
Alice                    Bob
  |  探测帧 ->              |
  |              <- 探测帧  |
  |                        |
  |-- 估计信道 h_AB         |-- 估计信道 h_BA
  |-- 量化 -> 密钥比特 k_A  |-- 量化 -> 密钥比特 k_B
  |                        |
  |  <- 信息协调 ->         |  (纠错使 k_A = k_B)
  |  <- 隐私放大 ->         |  (去除泄露信息)
  |                        |
  +-- 共享密钥 k ----------+
```

In [ ]:
# 信道特征密钥生成模拟
n_subcarriers = 64
h_true = np.random.randn(n_subcarriers) + 1j * np.random.randn(n_subcarriers)
h_A = h_true + 0.05 * (np.random.randn(n_subcarriers) + 1j * np.random.randn(n_subcarriers))
h_B = h_true + 0.05 * (np.random.randn(n_subcarriers) + 1j * np.random.randn(n_subcarriers))
h_E = np.random.randn(n_subcarriers) + 1j * np.random.randn(n_subcarriers)

print("=== 信道特征密钥生成 ===")
print(f"信道维度: {n_subcarriers}个子载波")
print(f"Alice-Bob信道相关性: {np.abs(np.corrcoef(np.abs(h_A), np.abs(h_B))[0,1]):.4f}")
print(f"Alice-Eve信道相关性: {np.abs(np.corrcoef(np.abs(h_A), np.abs(h_E))[0,1]):.4f}")

In [ ]:
# 量化: 信道特征 -> 密钥比特
def quantize_channel(h, method="magnitude"):
    if method == "magnitude":
        feature = np.abs(h)
    elif method == "phase":
        feature = np.angle(h)
    elif method == "complex":
        feature = np.concatenate([h.real, h.imag])
    bits = []
    threshold = np.median(feature)
    for f in feature:
        bits.append(1 if f > threshold else 0)
    return np.array(bits)

key_A_mag = quantize_channel(h_A, method="magnitude")
key_B_mag = quantize_channel(h_B, method="magnitude")
key_E_mag = quantize_channel(h_E, method="magnitude")
key_A_phase = quantize_channel(h_A, method="phase")
key_B_phase = quantize_channel(h_B, method="phase")

agree_AB_mag = np.mean(key_A_mag == key_B_mag)
agree_AB_phase = np.mean(key_A_phase == key_B_phase)
agree_AE = np.mean(key_A_mag == key_E_mag)

print(f"幅度量化 - Alice/Bob密钥一致率: {agree_AB_mag:.4f}")
print(f"相位量化 - Alice/Bob密钥一致率: {agree_AB_phase:.4f}")
print(f"幅度量化 - Alice/Eve密钥一致率: {agree_AE:.4f} (应接近0.5)")
print()
print("Alice/Bob一致率越高 -> 密钥生成越可靠")
print("Alice/Eve一致率越低 -> 密钥越安全")

In [ ]:
# 信息协调与隐私放大
print("=== 信息协调 (Information Reconciliation) ===")
print()
print("Alice和Bob的密钥可能有少量不一致 (信道估计噪声)")
print("需要通过公开信道交换纠错信息, 但不泄露密钥")
print()
print("方法:")
print("  1. Cascade协议: 交互式比特校验")
print("  2. LDPC/Turbo码: 发送校验子(syndrome)")
print("  3. BCH码: 纠错编码")
print()
print("隐私放大 (Privacy Amplification):")
print("  信息协调中泄露的比特需要去除")
print("  用哈希函数将长密钥压缩为短密钥")
print("  泄露l比特 -> 用(n - l)位通用哈希函数")
print()
print("密钥生成率 (Key Generation Rate):")
print("  R = I(A;B) - I(A;E) - 泄露量")
print("  I(A;B): Alice和Bob的互信息")
print("  I(A;E): Alice和Eve的互信息 (越小越安全)")

---
## 5. 认证协议设计

In [ ]:
# 射频指纹认证协议
print("=== 协议1: 基于指纹的设备认证 ===")
print("1. Device -> Gateway: 连接请求 + 发射信号")
print("2. Gateway: 采集IQ信号, 提取射频指纹")
print("3. Gateway: 指纹与数据库比对")
print("4. Gateway -> Device: 认证结果")
print("   - 匹配: 接入允许")
print("   - 不匹配: 拒绝接入")
print()
print("=== 协议2: 指纹 + 密码双因子认证 ===")
print("1. Device -> Gateway: 连接请求 + 密码")
print("2. Gateway: 验证密码 (第一因子)")
print("3. Gateway: 同时采集射频指纹 (第二因子)")
print("4. Gateway: 两个因子都通过才允许接入")
print("5. 任一因子失败 -> 触发安全告警")
print()
print("=== 协议3: 持续认证 (Continuous Authentication) ===")
print("传统: 只在连接建立时认证一次")
print("持续: 每个数据包都验证射频指纹")
print("优点: 即使初始认证被突破, 后续包也能检测异常")
print("挑战: 实时性要求高, 需要轻量模型")

In [ ]:
# 持续认证模拟
np.random.seed(42)
n_packets = 100
switch_point = 60

legit_embeds = np.random.randn(switch_point, 64) * 0.1 + np.random.randn(1, 64)
attack_embeds = np.random.randn(n_packets - switch_point, 64) * 0.1 + np.random.randn(1, 64) + 2
all_embeds = np.concatenate([legit_embeds, attack_embeds], axis=0)
legit_center = legit_embeds.mean(axis=0)
distances = np.linalg.norm(all_embeds - legit_center, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["blue"] * switch_point + ["red"] * (n_packets - switch_point)
axes[0].scatter(range(n_packets), distances, c=colors, s=10, alpha=0.7)
threshold = np.mean(distances[:switch_point]) + 2*np.std(distances[:switch_point])
axes[0].axhline(threshold, color="green", linestyle="--", label="Threshold")
axes[0].axvline(switch_point, color="orange", linestyle=":", alpha=0.5, label="Attack Starts")
axes[0].set_xlabel("Packet Index")
axes[0].set_ylabel("Distance to Legitimate Center")
axes[0].set_title("Continuous Authentication")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

window = 10
avg_dist = np.convolve(distances, np.ones(window)/window, mode="same")
axes[1].plot(avg_dist)
axes[1].axhline(threshold, color="green", linestyle="--", label="Threshold")
axes[1].axvline(switch_point, color="orange", linestyle=":", alpha=0.5, label="Attack Starts")
axes[1].set_xlabel("Packet Index")
axes[1].set_ylabel("Avg Distance (window=10)")
axes[1].set_title("Sliding Window Detection")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("持续认证: 攻击者接管后, 指纹距离骤增, 可快速检测")

---
## 6. 攻击与防御

In [ ]:
print("=== 攻击类型与防御 ===")
print()
print("1. 信号重放攻击 (Replay Attack)")
print("   攻击者: 录制合法设备的信号, 重新发送")
print("   防御:")
print("   - 随机挑战 (Challenge-Response): 网关发送随机数, 设备必须对随机数响应")
print("   - 时间戳: 检查信号的时间新鲜度")
print("   - 信道变化: 重放信号经过不同信道, 信道特征不同")
print()
print("2. 信号伪造攻击 (Signal Spoofing)")
print("   攻击者: 用SDR生成伪造信号, 模仿合法设备指纹")
print("   防御:")
print("   - 高维指纹: 单一特征可伪造, 高维组合难以同时伪造")
print("   - 细粒度特征: 深度学习提取的特征比手工特征更难伪造")
print("   - 多接收机: 不同位置接收, 交叉验证")
print()
print("3. 对抗样本攻击 (Adversarial Attack)")
print("   攻击者: 在信号上添加精心设计的微小扰动, 使分类器误判")
print("   防御:")
print("   - 对抗训练: 在训练中加入对抗样本")
print("   - 信号预处理: 滤波/量化去除对抗扰动")
print("   - 集成方法: 多模型投票")

In [ ]:
# 对抗样本攻击模拟 (FGSM)
model.eval()
x_test_tensor = torch.from_numpy(X_test[:20])
y_test_tensor = torch.from_numpy(y_test[:20])

with torch.no_grad():
    logits_clean, _ = model(x_test_tensor)
    pred_clean = logits_clean.argmax(1)
    acc_clean = (pred_clean == y_test_tensor).float().mean().item()

x_adv = x_test_tensor.clone().detach().requires_grad_(True)
logits_adv, _ = model(x_adv)
loss = nn.CrossEntropyLoss()(logits_adv, y_test_tensor)
loss.backward()
perturbation = 0.05 * x_adv.grad.sign()
x_attacked = x_test_tensor + perturbation

with torch.no_grad():
    logits_attacked, _ = model(x_attacked)
    pred_attacked = logits_attacked.argmax(1)
    acc_attacked = (pred_attacked == y_test_tensor).float().mean().item()

print(f"正常准确率: {acc_clean:.4f}")
print(f"FGSM攻击后准确率: {acc_attacked:.4f}")
print(f"扰动大小: epsilon=0.05 (人眼不可察觉的微小扰动)")
print()
print("对抗训练防御:")
print("  训练时混合正常样本和对抗样本")
print("  loss = CE(x_clean) + CE(x_adv)")

In [ ]:
# 对抗训练伪代码
print("=== 对抗训练 (Adversarial Training) ===")
print()
print("def adversarial_train(model, x, y, epsilon=0.01):")
print("    # 1. 生成对抗样本")
print("    x_adv = x.clone().detach().requires_grad_(True)")
print("    logits = model(x_adv)")
print("    loss = CE(logits, y)")
print("    loss.backward()")
print("    x_adv = x + epsilon * x_adv.grad.sign()")
print("    x_adv = torch.clamp(x_adv, x.min(), x.max())")
print()
print("    # 2. 在对抗样本上训练")
print("    logits_clean = model(x)")
print("    logits_adv = model(x_adv.detach())")
print("    loss = 0.5 * CE(logits_clean, y) + 0.5 * CE(logits_adv, y)")
print("    loss.backward()")
print("    optimizer.step()")
print()
print("其他防御方法:")
print("  - 输入预处理: 低通滤波/量化/压缩, 去除对抗扰动")
print("  - 特征压缩: 减少特征空间中的对抗方向")
print("  - 检测方法: 统计检验判断输入是否为对抗样本")
print("  - 认证专用: 结合信道特征, 对抗扰动改变信道特征可被检测")

---
## 7. 实际系统与标准

In [ ]:
print("=== IoT协议的物理层安全 ===")
print()
print("1. WiFi (802.11)")
print("   现有安全: WPA2/WPA3 (上层加密)")
print("   物理层安全研究:")
print("   - WiFi CSI (信道状态信息) 密钥生成")
print("   - WiFi设备射频指纹识别")
print("   - 802.11ay 引入波束成形的安全增强")
print()
print("2. Bluetooth/BLE")
print("   现有安全: 配对加密")
print("   物理层安全研究:")
print("   - BLE设备指纹识别")
print("   - 跳频模式分析")
print()
print("3. LoRa")
print("   现有安全: AES-128 (应用层)")
print("   物理层安全研究:")
print("   - LoRa设备指纹 (啁啾信号特征)")
print("   - 信道特征密钥生成")
print()
print("4. Zigbee")
print("   现有安全: AES-128 (网络层)")
print("   物理层安全研究:")
print("   - Zigbee设备指纹识别")
print("   - 轻量级认证协议")
print()
print("5. 5G/IoT-5G")
print("   现有安全: SIM卡+AKA协议")
print("   物理层安全研究:")
print("   - 5G射频指纹认证")
print("   - Massive MIMO信道安全")
print("   - 3GPP TR 33.836: 5G IoT安全增强")

In [ ]:
print("=== 实际部署考虑 ===")
print()
print("1. 注册阶段")
print("   - 安全环境: 在受控环境下采集指纹模板")
print("   - 多次采集: 取平均/中值, 提高模板质量")
print("   - 多SNR: 不同信噪比下采集, 提高鲁棒性")
print()
print("2. 认证延迟")
print("   - 采集时间: 通常1-10ms")
print("   - 推理时间: <1ms (轻量模型)")
print("   - 总延迟: <10ms (满足实时要求)")
print()
print("3. 设备数量扩展")
print("   - 小规模 (<100): 分类器直接分类")
print("   - 大规模 (>1000): 度量学习 + 检索")
print("   - 超大规模: 层次化认证 (先分组, 再组内识别)")
print()
print("4. 环境变化")
print("   - 位置变化: 指纹可能变化, 需要在线更新")
print("   - 温度变化: 硬件参数漂移, 需要周期性重注册")
print("   - 老化: 长期使用后指纹可能变化")
print()
print("5. 合规性")
print("   - 误判率要求: FAR < 0.1% (安全场景)")
print("   - 拒判率要求: FRR < 5% (用户体验)")
print("   - EER (等错误率): 系统整体安全指标")

In [ ]:
print("=== 物理层安全研究前沿 ===")
print()
print("1. 联邦学习 + 射频指纹")
print("   多个网关协同训练, 不共享原始数据")
print("   隐私保护 + 性能提升")
print()
print("2. 可解释性")
print("   深度学习指纹的物理可解释性")
print("   哪些硬件缺陷被模型捕获?")
print()
print("3. 跨协议指纹")
print("   同一设备在不同协议下的指纹一致性")
print("   WiFi指纹能否用于识别同一设备的蓝牙?")
print()
print("4. 量子安全")
print("   量子计算对物理层安全的影响")
print("   量子密钥分发 (QKD) 与物理层密钥生成的对比")
print()
print("5. RIS辅助的物理层安全")
print("   可重构智能表面 (RIS) 增强信道安全")
print("   主动调控信道, 增大合法/窃听信道差异")
print()
print("6. AI安全")
print("   对抗攻击/后门攻击对RFFI的威胁")
print("   AI模型的鲁棒性认证")
print()
print("7. 标准化")
print("   IEEE 802.11bf (WiFi感知): 信道特征标准化")
print("   3GPP RAN: 5G-Advanced物理层安全")
print("   ETSI: IoT安全标准")

---
## 总结

### 物理层安全知识图谱

```
物理层安全 -- IoT设备认证
|-- 基础原理
|   |-- 互易性: h_AB = h_BA
|   |-- 空间唯一性: h_AE != h_AB
|   +-- 硬件唯一性: 每个设备指纹不同
|
|-- 射频指纹认证
|   |-- Closed-set: 分类问题
|   |-- Open-set: 异常检测 (关键!)
|   |-- 度量学习: ArcFace + Center Loss
|   +-- 持续认证: 每个包都验证
|
|-- 信道特征密钥生成
|   |-- 信道探测 -> 量化 -> 信息协调 -> 隐私放大
|   +-- 密钥生成率: R = I(A;B) - I(A;E) - 泄露
|
|-- 攻击与防御
|   |-- 重放攻击 -> 随机挑战/时间戳
|   |-- 伪造攻击 -> 高维指纹/多接收机
|   +-- 对抗攻击 -> 对抗训练/信号预处理
|
+-- 实际系统
    |-- WiFi/BLE/LoRa/Zigbee/5G
    |-- 延迟 < 10ms, FAR < 0.1%
    +-- 联邦学习/可解释性/标准化
```

### 关键指标

| 指标 | 含义 | 安全要求 |
|------|------|----------|
| GAR | 合法设备接受率 | > 95% |
| FAR | 非法设备接受率 | < 0.1% |
| EER | 等错误率 | 越低越好 |
| AUROC | ROC曲线下面积 | > 0.999 |
| 密钥一致率 | Alice/Bob密钥匹配率 | > 99% |
| 密钥熵 | 密钥随机性 | 接近1 bit/bit |